In [107]:
import sqlite3
import csv
import json
from datetime import datetime

In [1]:
class BankAccount:
    def __init__(self,customer_name,account_type,balance,created_date):
        self.customer_name=customer_name
        self.account_type=account_type
        self.balance=balance
        self.created_date=created_date
    def __str__(self):
        return f"BankAccount=( {self.customer_name=} | {self.account_type=} | {self.balance=} | {self.created_date=} ) "
    def to_dict(self):
        return{
            "customer_name": self.customer_name,
            "account_type": self.account_type,
            "balance": self.balance,
            "created_date": self.created_date
        }
    @staticmethod
    def from_dict(data):
        return BankAccount(
            data["customer_name"],
            data["account_type"],
            data["balance"],
            data["created_date"]
        )

In [20]:
b1=BankAccount("john","Saving",1000,"2002-09-11")
a=b1.to_dict()
a

{'customer_name': 'john',
 'account_type': 'Saving',
 'balance': 1000,
 'created_date': '2002-09-11'}

In [21]:

print(b1.from_dict(a))


BankAccount=( self.customer_name='john' | self.account_type='Saving' | self.balance=1000 | self.created_date='2002-09-11' ) 


In [109]:
def conncet_db():
    return sqlite3.connect("bankk.db")
def createtable():
    conn=conncet_db()
    cursor=conn.cursor()
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS accounts(
                account_id INTEGER PRIMARY KEY AUTOINCREMENT,
                customer_name TEXT,
                account_type TEXT,
                balance REAL,
                created_date TEXT)
    """)
    conn.commit()
    conn.close()

In [110]:
def validate_account_type(account_type):
    return account_type in ["Savings","Current"]
    
def validate_amount(balance):
    return balance>0
    
def validate_date(created_date):
    try:
        datetime.strptime(created_date,"%Y-%m-%d")
        return True
    except:
        return False



In [111]:
def create_account():
    customer_name=input("enter the customer_name: ")
    account_type=input("enter the account_type [Savings or Current]: ")
    balance=float(input("enter the balance: "))
    created_date=input("enter the created_date[YYYY-MM-DD]: ")

    if not customer_name or not validate_account_type(account_type) or not validate_amount(balance) or not validate_date(created_date):
        print("invalid input")
        return
    conn=conncet_db()
    cursor=conn.cursor()
    cursor.execute("INSERT INTO accounts(customer_name,account_type,balance,created_date) VALUES(?,?,?,?) ",
                  (customer_name,account_type,balance,created_date))
    conn.commit()
    conn.close()
    print("account created sucessfuly...")
        


In [112]:
def view_accounts():
    conn=conncet_db()
    cursor=conn.cursor()
    cursor.execute("SELECT * FROM accounts ")
    rows=cursor.fetchall()
    for row in rows:
        print(row)
    conn.close()



In [113]:
def update_account():
    customer_name=input("enter the customer_name: ")
    account_type=input("enter the account_type [Savings or Current]: ")
    account_id=int(input("enter account_id: "))

    if not customer_name or not validate_account_type(account_type):
        print("invalid input")
        return
    conn=conncet_db()
    cursor=conn.cursor()
    cursor.execute("UPDATE accounts SET customer_name=? ,account_type=? WHERE account_id=?",(customer_name,account_type,account_id))
    conn.commit()
    conn.close()
    print("UPDATED SUCESSFYLLY...")

In [129]:
def delete_account():
    account_id=int(input("enter account_id: "))
    conn=conncet_db()
    cursor=conn.cursor()
    cursor.execute("DELETE FROM accounts WHERE account_id=? ",(account_id,))
    conn.commit()
    conn.close()
    print("DELETED SUCESSFYLLY...")

In [115]:
def deposit():
    account_id=int(input("enter account_id: "))
    balance=float(input("enter the balance to deposite: "))
    if not validate_amount(balance):
        print("amount need to be greater than zero..")
        return

    conn=conncet_db()
    cursor=conn.cursor()
    cursor.execute("SELECT balance FROM accounts WHERE account_id=?",(account_id,))
    result=cursor.fetchone()
    if result:
        new_bal=result[0]+balance
        cursor.execute("UPDATE accounts SET balance=? WHERE account_id=?",(new_bal,account_id))
        conn.commit()
        conn.close()
        print("DEPOSITED SUCESSFYLLY...")
    else:
        print("Account not found...")

In [116]:
def withdraw():
    account_id=int(input("enter account_id: "))
    balance=float(input("enter the balance to deposite: "))
    if not validate_amount(balance):
        print("amount need to be greater than zero..")
        return
    conn=conncet_db()
    cursor=conn.cursor()
    cursor.execute("SELECT balance FROM accounts WHERE account_id=? ",(account_id,))
    result=cursor.fetchone()
    if result:
        if result[0]>=balance:
            new_bal=result[0]-balance
            cursor.execute("UPDATE accounts SET balance=? WHERE account_id=? ",(new_bal,account_id))
            conn.commit()
            conn.close()
            print("withdraw SUCESSFYLLY...")
        else:
            print("Insufficient balance!")
    else:
        print("Account not found...")

In [124]:
def search():
    print("searchoptions: \n ByAccountID \n ByCustomerName(partialmatch) \n ByAccountType")
    choice=int(input("enter the choice: "))
    conn=conncet_db()
    cursor=conn.cursor()
    
    if choice==1:
        account_id=int(input("enter account_id: "))
        cursor.execute("SELECT * FROM accounts WHERE account_id=?",(account_id,))
    elif choice==2:
        customer_name=input("enter the customer_name: ")
        cursor.execute("SELECT * FROM accounts WHERE customer_name LIKE ?",("%" +customer_name+"%",))
    elif choice==3:
        account_type=input("enter the account_type [Savings or Current]: ")
        if not validate_account_type(account_type):
            return
        cursor.execute("SELECT * FROM accounts WHERE account_type=?",(account_type,))
    else:
        print("INVALID CHOICE")

    rows=cursor.fetchall()
    if rows:  
        for row in rows:
            print(row)
    else:
         print("No records found")
    conn.close()    

In [127]:
def export_csv():
    conn=conncet_db()
    cursor=conn.cursor()
    cursor.execute("SELECT * FROM accounts")
    rows=cursor.fetchall()
    
    with open("accounts.csv","w",newline='') as f:
        writer=csv.writer(f)
        writer.writerow(["id","name","type","balance","date"])
        writer.writerows(rows)
        
    conn.close()

In [128]:
def export_json():
    conn=conncet_db()
    conn.row_factory = sqlite3.Row 
    cursor=conn.cursor()
    cursor.execute("SELECT customer_name,account_type,balance,created_date FROM accounts")
    
    data=[dict(row) for row in cursor.fetchall()]
    with open("accounts.json","w") as f:
        json.dump(data,f)
    conn.close()

In [130]:
def menu():
    createtable()

    while True:
        print("\n1.Create 2.View 3.Search 4.Update 5.Delete")
        print("6.Deposit 7.Withdraw 8.Export CSV 9.Export JSON 10.Exit")
        
        try:
            
            choice = int(input("Enter choice: "))
            if choice==1:
                create_account()
            elif choice==2:
                view_accounts()
            elif choice==3:
                search()
            elif choice==4:
                update_account()
            elif choice==5:
                delete_account()
            elif choice==6:
                deposit()
            elif choice==7:
                withdraw()
            elif choice==8:
                export_csv()
            elif choice==9:
                export_json()
            elif choice==10:
                print("loged out...")
                break
            else:
                print("invalid choice")
        except Exception as e:
            print("Error",e)
            

In [131]:
menu()


1.Create 2.View 3.Search 4.Update 5.Delete
6.Deposit 7.Withdraw 8.Export CSV 9.Export JSON 10.Exit


Enter choice:  5
enter account_id:  2


DELETED SUCESSFYLLY...

1.Create 2.View 3.Search 4.Update 5.Delete
6.Deposit 7.Withdraw 8.Export CSV 9.Export JSON 10.Exit


Enter choice:  8



1.Create 2.View 3.Search 4.Update 5.Delete
6.Deposit 7.Withdraw 8.Export CSV 9.Export JSON 10.Exit


Enter choice:  9



1.Create 2.View 3.Search 4.Update 5.Delete
6.Deposit 7.Withdraw 8.Export CSV 9.Export JSON 10.Exit


Enter choice:  10


loged out...
